# 01 データ探索（EDA）

楽天APIから取得した返礼品データと総務省「ふるさと納税に関する現況調査」データの全体像を把握する。

**分析の目的**
- データ品質の確認（欠損・外れ値・重複）
- 価格帯・レビュー数・カテゴリの分布把握
- 総務省データとの突合可能性の確認
- Week 2 の4切り口分析に向けた仮説の形成

In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.1f}'.format)

DATA_DIR = Path('../data')

## 1. データ読み込み

In [3]:
# 楽天データ
df = pd.read_parquet(DATA_DIR / 'processed/products_raw.parquet')
print(f'楽天データ: {len(df):,} 件')
df.head(3)

楽天データ: 32,244 件


,item_code,genre_id,item_price,review_count,review_average,shop_name,item_caption_snippet,item_name,search_keyword
0,f032051-hanamaki:10000827,110435,8000,13489,4.5,岩手県花巻市,★★内容量と配送時期をお選びいただけます★★ 内容量：1kg【スピード配送対応！選べる配送月...,【ふるさと納税】 ＼総合1位常連 高リピート率＆衝撃の厚み10mm ／厚切り牛タン塩味 50...,ふるさと納税 牛肉
1,f253847-ryuo:10001383,112666,7000,13326,4.8,滋賀県竜王町,商品説明内容量 合計1~12kg(200g×5~30個入) 200g/1個 賞味期限 製造日...,【ふるさと納税】 ＼総合1位獲得／ 近江牛 入り ハンバーグ 6kg 3kg 2kg 1kg...,ふるさと納税 牛肉
2,f016918-betsukai:10002938,112666,10000,12137,4.4,北海道別海町,【父の日】ギフト対応について ▼ギフトの種類 父の日ギフト：6/18～21までお届け（6/8...,【ふるさと納税】総合1位 北海道産 牛肉 ふるさと納税 別海牛 焼肉 （ ふるさと納税 肉 ...,ふるさと納税 牛肉


In [4]:
# 総務省データ
from src.api.soumu_loader import load_soumu_csv

df_soumu = load_soumu_csv(DATA_DIR / 'raw/soumu_donations.csv')
print(f'総務省データ: {len(df_soumu):,} 自治体')
df_soumu.head(3)

2026-05-05 13:38:05 | INFO     | src.api.soumu_loader:56 - CSVロード: 1890 行 × 33 列
2026-05-05 13:38:05 | INFO     | src.api.soumu_loader:89 - 総務省データ読み込み完了 (令和４年度): 1,741 自治体


総務省データ: 1,741 自治体


,都道府県名,市区町村名,受入額_合計,受入件数
0,北海道,札幌市,1741318,63070
1,北海道,函館市,1197337,66872
2,北海道,小樽市,888995,55914


## 2. 楽天データの品質確認

In [6]:
print('=== データ型 ===')
print(df.dtypes)
print()
print('=== 欠損値 ===')
print(df.isnull().sum())
print()
print('=== 基本統計 ===')
df[['item_price', 'review_count', 'review_average']].describe()

=== データ型 ===
item_code                   str
genre_id                  int64
item_price                int64
review_count              int64
review_average          float64
shop_name                   str
item_caption_snippet        str
item_name                   str
search_keyword              str
dtype: object

=== 欠損値 ===
item_code               0
genre_id                0
item_price              0
review_count            0
review_average          0
shop_name               0
item_caption_snippet    0
item_name               0
search_keyword          0
dtype: int64

=== 基本統計 ===


,item_price,review_count,review_average
count,"32,244.0","32,244.0","32,244.0"
mean,"51,292.5",34.2,3.5
std,"231,847.3",316.9,2.0
min,"1,000.0",0.0,0.0
25%,"11,000.0",1.0,3.0
50%,"15,000.0",4.0,4.5
75%,"30,000.0",11.0,4.9
max,"11,360,000.0","30,508.0",5.0


In [5]:
# キーワード別件数
keyword_counts = df['search_keyword'].value_counts().reset_index()
keyword_counts.columns = ['キーワード', '件数']

fig = px.bar(
    keyword_counts,
    x='件数', y='キーワード',
    orientation='h',
    title='検索キーワード別 取得件数',
    color='件数',
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 3. 価格帯分布

In [ ]:
# 外れ値除外（上位1%）して分布確認
price_99 = df['item_price'].quantile(0.99)  # 下位から99%の範囲のデータを利用する。
df_plot = df[df['item_price'] <= price_99]

fig = px.histogram(
    df_plot,
    x='item_price',
    nbins=50,
    title=f'返礼品 価格帯分布（上位1%除外、N={len(df_plot):,}件）',
    labels={'item_price': '寄付金額（円）'},
    color_discrete_sequence=['#e84393']
)
fig.add_vline(x=10000, line_dash='dash', line_color='gray', annotation_text='1万円')
fig.add_vline(x=30000, line_dash='dash', line_color='gray', annotation_text='3万円')
fig.add_vline(x=50000, line_dash='dash', line_color='gray', annotation_text='5万円')
fig.show()

# 価格帯別件数
bins = [0, 5000, 10000, 20000, 30000, 50000, 100000, float('inf')]
labels = ['〜5千円', '5千〜1万円', '1〜2万円', '2〜3万円', '3〜5万円', '5〜10万円', '10万円〜']
df['price_segment'] = pd.cut(df['item_price'], bins=bins, labels=labels)
print(df['price_segment'].value_counts().sort_index())

price_segment
〜5千円        967
5千〜1万円     6725
1〜2万円     13119
2〜3万円      3852
3〜5万円      2844
5〜10万円     2268
10万円〜      2469
Name: count, dtype: int64


## 4. レビュー分布

In [8]:
# レビューあり商品のみ
df_reviewed = df[df['review_count'] > 0]
print(f'レビューあり: {len(df_reviewed):,} 件 ({len(df_reviewed)/len(df)*100:.1f}%)')
print(f'レビューなし: {len(df)-len(df_reviewed):,} 件')

fig = px.scatter(
    df_reviewed[df_reviewed['item_price'] <= price_99],
    x='item_price',
    y='review_count',
    color='review_average',
    color_continuous_scale='RdYlGn',
    title='価格 × レビュー数（色: 評価平均）',
    labels={
        'item_price': '寄付金額（円）',
        'review_count': 'レビュー数',
        'review_average': '評価平均'
    },
    opacity=0.5,
    hover_data=['shop_name']
)
fig.show()

レビューあり: 25,143 件 (78.0%)
レビューなし: 7,101 件


In [9]:
# 評価平均の分布
fig = px.histogram(
    df_reviewed,
    x='review_average',
    nbins=20,
    title='レビュー評価平均の分布',
    labels={'review_average': '評価平均（5点満点）'},
    color_discrete_sequence=['#00b900']
)
fig.show()

print('評価平均の統計:')
print(df_reviewed['review_average'].describe())

評価平均の統計:
count   25,143.0
mean         4.5
std          0.6
min          1.0
25%          4.3
50%          4.7
75%          5.0
max          5.0
Name: review_average, dtype: float64


## 5. キーワード別 価格・レビュー比較

In [10]:
# キーワードのラベルを短縮
df['category_label'] = df['search_keyword'].str.replace('ふるさと納税 ', '')

# カテゴリ別 中央値価格
cat_stats = df.groupby('category_label').agg(
    件数=('item_price', 'count'),
    価格中央値=('item_price', 'median'),
    価格平均=('item_price', 'mean'),
    レビュー数中央値=('review_count', 'median'),
    評価平均=('review_average', lambda x: x[x > 0].mean())
).reset_index()

cat_stats = cat_stats.sort_values('価格中央値', ascending=False)
cat_stats

,category_label,件数,価格中央値,価格平均,レビュー数中央値,評価平均
2,家電,2465,"67,000.0","224,449.1",0.0,4.3
3,旅行,2668,"50,000.0","116,735.5",1.0,4.5
0,カニ,2356,"29,000.0","93,600.1",0.0,4.0
1,ホタテ,1679,"19,000.0","39,364.3",0.0,4.5
4,日用品,2607,"16,000.0","30,814.5",2.0,4.6
6,牛肉,2854,"15,000.0","21,677.6",10.0,4.5
9,酒,2014,"15,000.0","19,380.9",4.0,4.7
7,米,2416,"14,000.0","21,648.8",7.0,4.7
5,果物,2915,"13,000.0","15,864.7",11.0,4.3
8,豚肉,2520,"13,000.0","16,836.0",3.0,4.6


In [11]:
fig = px.bar(
    cat_stats,
    x='category_label', y='価格中央値',
    title='カテゴリ別 寄付金額 中央値',
    labels={'category_label': 'カテゴリ', '価格中央値': '中央値（円）'},
    color='価格中央値',
    color_continuous_scale='Oranges'
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

In [12]:
fig = px.scatter(
    cat_stats,
    x='価格中央値',
    y='レビュー数中央値',
    size='件数',
    color='評価平均',
    text='category_label',
    color_continuous_scale='RdYlGn',
    title='カテゴリ別 価格中央値 × レビュー数中央値（バブルサイズ: 件数）',
    labels={
        '価格中央値': '寄付金額 中央値（円）',
        'レビュー数中央値': 'レビュー数 中央値',
    }
)
fig.update_traces(textposition='top center')
fig.show()

## 6. 総務省データの確認

In [13]:
print('=== 欠損値 ===')
print(df_soumu.isnull().sum())
print()
print('=== 受入額統計（千円）===')
print(df_soumu['受入額_合計'].describe())
print()
# 都道府県別 受入額合計トップ10
pref_total = df_soumu.groupby('都道府県名')['受入額_合計'].sum().sort_values(ascending=False)
print('都道府県別 受入額合計 TOP10（千円）:')
print(pref_total.head(10))

=== 欠損値 ===
都道府県名     0
市区町村名     0
受入額_合計    0
受入件数      0
dtype: int64

=== 受入額統計（千円）===
count        1,741.0
mean       548,029.5
std      1,325,852.5
min              0.0
25%         45,709.0
50%        169,301.0
75%        497,240.0
max     19,592,615.0
Name: 受入額_合計, dtype: float64

都道府県別 受入額合計 TOP10（千円）:
都道府県名
北海道     144732485
福岡県      55063249
宮崎県      46541720
鹿児島県     42415873
佐賀県      40832197
山形県      38138542
静岡県      32855574
大阪府      32011204
山梨県      31521985
新潟県      30436324
Name: 受入額_合計, dtype: int64


In [14]:
# 自治体別受入額 TOP20
top20 = df_soumu.nlargest(20, '受入額_合計').copy()
top20['自治体'] = top20['都道府県名'] + ' ' + top20['市区町村名']
top20['受入額_億円'] = top20['受入額_合計'] / 100_000  # 千円→億円

fig = px.bar(
    top20,
    x='受入額_億円', y='自治体',
    orientation='h',
    title='ふるさと納税 受入額 自治体TOP20（令和4年度）',
    labels={'受入額_億円': '受入額（億円）', '自治体': ''},
    color='受入額_億円',
    color_continuous_scale='Reds'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show()

## 7. 楽天データの出店自治体確認

In [15]:
# shop_name の例を確認（自治体名との突合に使う）
print('ショップ名サンプル（上位20件）:')
print(df['shop_name'].value_counts().head(20))

ショップ名サンプル（上位20件）:
shop_name
宮崎県都城市     565
北海道根室市     516
北海道別海町     351
大阪府泉佐野市    308
京都府京都市     304
宮城県大河原町    246
宮城県角田市     236
北海道釧路市     201
福岡県糸島市     195
福岡県久留米市    174
京都府京丹後市    173
岐阜県高山市     165
山形県山形市     163
秋田県北秋田市    158
宮城県気仙沼市    154
山口県下関市     154
北海道弟子屈町    152
静岡県焼津市     151
宮崎県宮崎市     148
静岡県富士市     146
Name: count, dtype: int64


In [16]:
# ユニーク出店数
print(f'ユニークショップ数: {df["shop_name"].nunique():,}')
print(f'ユニークitem_code数: {df["item_code"].nunique():,}')

# ショップ別商品数TOP20
shop_top20 = df['shop_name'].value_counts().head(20).reset_index()
shop_top20.columns = ['ショップ名', '商品数']

fig = px.bar(
    shop_top20,
    x='商品数', y='ショップ名',
    orientation='h',
    title='出店数の多いショップ（自治体）TOP20',
    color='商品数',
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=550)
fig.show()

ユニークショップ数: 1,497
ユニークitem_code数: 32,244


## 8. EDA まとめ・仮説メモ

### データ品質
- 楽天データ: 32,244件、重複除去済み
- 総務省データ: 1,741自治体（令和4年度）

### 主な発見（暫定）
1. **価格帯の集中**: 1万円前後に最も多くの商品が集中する「心理的価格ゾーン」が存在
2. **カテゴリ差**: 家電・旅行は高単価、食品系は低〜中単価
3. **レビュー数の偏り**: 上位少数の商品にレビューが集中（ロングテール構造）
4. **評価の高さ**: 評価平均4.5前後が多く「返礼品への高満足度」を示唆

### Week 2 分析への仮説
- 食品（特に肉・魚介）は低価格でレビュー数が多い「コスパ重視型」
- 家電・旅行は高単価でレビュー数は少ない「高額選択型」
- 還元率が高いカテゴリほどレビュー評価も高い可能性